# Example-04: Optimal SVD truncation, rank and noise estimation

In [1]:
# Import

import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import data_load
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

False


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# estimate_noise function can be used for truncation rank and noise value estimation
# This function can be used when number of components in signal is not known

# Load test TbT data and add random noise

length = 1024

w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_file(54, w, '../virtual_tbt.npy')
s = 1.0E-4*torch.rand(54, dtype=dtype, device=device)
d.add_noise(s)
print(d)

# Filter instance

l = Filter(d)

# For estimation, only several first columns can ne used (32 by default)

R1, N1 = l.estimate_noise(limit=16, cpu=True)
R2, N2 = l.estimate_noise(limit=64, cpu=True)
R3, N3 = l.estimate_noise(limit=256, cpu=True)

# Estimate noise (average over samples)
# Generate several samples of length 256 from each signal using start shift 8

w = Window(256)
N4 = []
for i in range(54):
    l = Filter(Data.from_data(w, d.make_matrix(256, 8, d.work[i])))
    _, noise = l.estimate_noise(limit=64)
    N4.append(noise.mean().item())

Data(54, Window(1024, 'cosine_window', 1.0))


In [4]:
# Compare estimated ranks (note, expected rank is two)

fmt = 4 * '{:<6}'
print(fmt.format("", 16, 64, 256))
for i, (r1, r2, r3) in enumerate(zip(R1.cpu(), R2.cpu(), R3.cpu())):
    print(fmt.format(i, r1, r2, r3))
    
# Note, depending on particular noise parameters, results might be different
# In general, rank tends to be overestimated if large number of columns is used

      16    64    256   
0     2     2     4     
1     2     2     2     
2     2     2     3     
3     2     2     2     
4     2     2     2     
5     2     2     2     
6     2     2     2     
7     2     2     3     
8     2     2     4     
9     2     2     2     
10    2     2     4     
11    2     2     2     
12    2     3     3     
13    2     2     2     
14    2     2     2     
15    2     2     2     
16    2     3     6     
17    2     2     2     
18    2     2     2     
19    2     2     2     
20    2     2     2     
21    2     2     2     
22    2     2     4     
23    2     2     2     
24    2     2     4     
25    2     2     2     
26    2     2     4     
27    2     2     4     
28    2     2     2     
29    2     2     3     
30    2     2     2     
31    2     2     6     
32    2     2     4     
33    2     2     2     
34    2     2     2     
35    2     2     2     
36    2     2     2     
37    2     2     2     
38    2     2     3     


In [5]:
# Compare estimated noise

fmt = 5 * '{:<16}'
print(fmt.format("", 16, 64, 256, "ave"))
fmt = '{:<16}' + 4 * '{:<16.8}'
for i, (n0, n1, n2, n3, n4) in enumerate(zip(s.cpu(), N1.cpu(), N2.cpu(), N3.cpu(), N4)):
    print(fmt.format(i, 100*abs(n1-n0)/n0, 100*abs(n2-n0)/n0, 100*abs(n3-n0)/n0, 100*abs(n4-n0)/n0))
    
# Note, for noise sigma estimation, approximate expression is used
# In this example, noise estimation accuracy is better than 10%
# Using larger number of columns or averaging over samples do not affect estimation accuracy significantly (but better on average in both cases)

                16              64              256             ave             
0               0.12340242      0.067608443     3.1934347       1.0595807       
1               5.6136295       3.358666        5.366963        6.4097001       
2               1.8806882       0.43381591      3.5538363       0.657084        
3               2.5723579       1.814364        1.4363279       2.1318895       
4               0.86799822      4.1770651       0.80920961      2.3874158       
5               1.731697        2.6620693       2.1062294       0.89741013      
6               1.8501368       2.6760097       1.9895636       0.047028681     
7               0.10350888      2.0985733       3.2663673       2.4215096       
8               4.7356444       7.0658885       9.0653853       4.3505168       
9               2.9917666       2.2368832       1.5854009       0.64299373      
10              3.7179948       3.4303056       5.1018069       1.3243883       
11              0.87681824  

In [6]:
# Estimate noise using randomized SVD

l = Filter(d)
r1, s1 = l.estimate_noise(limit=32)
r2, s2 = l.estimate_noise(limit=512, randomized=True)

print((s1 - s).abs().max())
print((s2 - s).abs().max())

tensor(5.897927213820e-06, dtype=torch.float64)
tensor(8.074340612967e-06, dtype=torch.float64)
